# 09 — Performance: torch.compile, torch.export, Profiling, and Memory

Goal: make models fast, stable, and deployable.

_Generated: 2026-01-25_

## Setup

```bash
pip install torch torchvision torchaudio
pip install transformers datasets tokenizers accelerate evaluate
pip install matplotlib tensorboard
```

In [ ]:

import os, math, random
import numpy as np
import torch

def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

device = get_device()
print("torch:", torch.__version__)
print("device:", device)

## 1. torch.compile

`torch.compile` can accelerate training/inference by graph compilation.
Best results: stable shapes, minimal Python branching in hot paths.

In [ ]:

import torch, torch.nn as nn
import torch.nn.functional as F

class TinyMLP(nn.Module):
    def __init__(self, d=256):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d, d*2), nn.GELU(), nn.Linear(d*2, d))
    def forward(self, x):
        return self.net(x)

m = TinyMLP().to(device).eval()
x = torch.randn(2048, 256, device=device)

if hasattr(torch, "compile"):
    cm = torch.compile(m)
    with torch.inference_mode():
        y1 = m(x)
        y2 = cm(x)
    print("compiled max diff:", (y1-y2).abs().max().item())
else:
    print("torch.compile not available.")

## 2. torch.export

Captures an ExportedProgram graph for deployment/transformations.

In [ ]:

import torch

def try_export():
    if not hasattr(torch, "export"):
        print("torch.export not available.")
        return None
    m = TinyMLP().to(device).eval()
    example = (torch.randn(2,256, device=device),)
    try:
        ep = torch.export.export(m, example)
        print("ExportedProgram ok.")
        return ep
    except Exception as e:
        print("export failed:", type(e).__name__, str(e)[:200], "...")
        return None

ep = try_export()

## 3. Profiling with torch.profiler

In [ ]:

import torch.profiler as profiler

m = TinyMLP().to(device)
inp = torch.randn(4096, 256, device=device)

acts = [profiler.ProfilerActivity.CPU]
if torch.cuda.is_available():
    acts.append(profiler.ProfilerActivity.CUDA)

with profiler.profile(activities=acts, record_shapes=True) as prof:
    for _ in range(50):
        _ = m(inp)

print(prof.key_averages().table(sort_by="cpu_time_total", row_limit=10))

## 4. Checklist

- inference_mode for eval
- AMP for CUDA
- pinned memory + non_blocking transfers
- dataloader worker tuning
- activation checkpointing for large models